# 02_02 · Preprocesado — CNC Mill Tool Wear


In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

PROCESSED = '../../data/processed'

In [2]:
cnc_raw = pd.read_parquet(f'{PROCESSED}/cnc_raw.parquet')

print('CNC raw shape:', cnc_raw.shape)
cnc_raw.head(3)

CNC raw shape: (25286, 55)


,X1_ActualPosition,X1_ActualVelocity,X1_ActualAcceleration,X1_CommandPosition,X1_CommandVelocity,X1_CommandAcceleration,X1_CurrentFeedback,X1_DCBusVoltage,X1_OutputCurrent,X1_OutputVoltage,...,M1_sequence_number,M1_CURRENT_FEEDRATE,Machining_Process,experiment,material,feedrate,clamp_pressure,tool_condition,machining_finalized,passed_visual_inspection
0,198.0,0.0,0.00,198.0,0.0,0.000000,0.18,0.0207,329.0,2.77,...,0.0,50.0,Starting,1,wax,6,4.0,unworn,yes,yes
1,198.0,-10.8,-350.00,198.0,-13.6,-358.000000,-10.90,0.1860,328.0,23.30,...,4.0,50.0,Prep,1,wax,6,4.0,unworn,yes,yes
2,196.0,-17.8,-6.25,196.0,-17.9,-0.000095,-8.59,0.1400,328.0,30.60,...,7.0,50.0,Prep,1,wax,6,4.0,unworn,yes,yes


In [3]:
# Columnas de sensores (excluir columnas de control y metadata)
exclude = ['experiment', 'Machining_Process', 'M1_CURRENT_PROGRAM_NUMBER',
           'M1_sequence_number', 'material', 'feedrate', 'clamp_pressure',
           'tool_condition', 'machining_finalized', 'passed_visual_inspection']
sensor_cols = [c for c in cnc_raw.columns if c not in exclude]

# Agregar por experimento: media, std, max, min de cada señal
agg_dict = {c: ['mean', 'std', 'max', 'min'] for c in sensor_cols}
cnc_agg = cnc_raw.groupby('experiment').agg(agg_dict)
cnc_agg.columns = ['_'.join(c) for c in cnc_agg.columns]
cnc_agg = cnc_agg.reset_index()

# Recuperar metadata desde cnc_raw (ya incluida en el parquet)
meta_cols = ['experiment', 'material', 'feedrate', 'clamp_pressure', 'tool_condition']
meta = cnc_raw[[c for c in meta_cols if c in cnc_raw.columns]].drop_duplicates('experiment')
cnc_agg = cnc_agg.merge(meta, on='experiment', how='left')

# Encoding y target
cnc_agg['material_enc'] = LabelEncoder().fit_transform(cnc_agg['material'])
cnc_agg['target'] = (cnc_agg['tool_condition'] == 'worn').astype(int)

print('CNC agregado shape:', cnc_agg.shape)
print(f'worn: {cnc_agg["target"].sum()} | unworn: {(cnc_agg["target"]==0).sum()}')

CNC agregado shape: (18, 187)
worn: 10 | unworn: 8


## 4. CNC — Split

Con **18 muestras** en total (10 worn, 8 unworn), cualquier split fijo 80/20 deja
solo 3-4 muestras en test — demasiado poco para estimar métricas fiables.

**Solución:** LeaveOneOut Cross-Validation (LOO-CV):
- En cada iteración, entrenamos con 17 muestras y evaluamos con 1
- Repetimos 18 veces → cada muestra actúa como test exactamente una vez
- Las métricas promediadas sobre las 18 iteraciones son más estables que un único split

También guardamos un split 80/20 para comparaciones rápidas, pero LOO-CV
es la referencia de evaluación para CNC.


In [4]:
from sklearn.model_selection import LeaveOneOut

feature_cols_cnc = [c for c in cnc_agg.columns
                    if c not in ['experiment', 'material', 'tool_condition', 'target']]

X_cnc = cnc_agg[feature_cols_cnc].fillna(0)
y_cnc = cnc_agg['target']

# Split primero (estratificado), escalar después: el scaler solo ve train
X_train_raw, X_test_raw, y_train_c, y_test_c = train_test_split(
    X_cnc, y_cnc, test_size=0.2, random_state=42, stratify=y_cnc)

scaler_cnc = StandardScaler().fit(X_train_raw)
X_train_c = scaler_cnc.transform(X_train_raw)
X_test_c  = scaler_cnc.transform(X_test_raw)

print(f'CNC features: {X_cnc.shape[1]}')
print(f'CNC Train: {X_train_c.shape} | Test: {X_test_c.shape}')
print('LOO-CV se aplicará en el notebook 03 (evaluación más robusta con 18 muestras).')


CNC features: 183
CNC Train: (14, 183) | Test: (4, 183)
LOO-CV se aplicará en el notebook 03 (evaluación más robusta con 18 muestras).


## 5. Guardar datos procesados CNC

Guardamos `cnc_aggregated.csv` y actualizamos `splits.pkl` con las claves `cnc` y `cnc_loo`.


In [5]:
import pickle, os
os.makedirs('../../data/processed', exist_ok=True)

pkl_path = '../../data/processed/splits.pkl'
splits = {}
if os.path.exists(pkl_path):
    with open(pkl_path, 'rb') as f:
        splits = pickle.load(f)

splits['cnc'] = (X_train_c, X_test_c, y_train_c, y_test_c)
# Para LOO se guardan las features SIN escalar: el escalado va dentro del
# Pipeline de cada modelo y se ajusta en cada fold (sin leakage)
splits['cnc_loo'] = (X_cnc.values, y_cnc)
with open(pkl_path, 'wb') as f:
    pickle.dump(splits, f)

cnc_agg.to_csv('../../data/processed/cnc_aggregated.csv', index=False)
print('CNC guardado OK.')
print(f'Train: {X_train_c.shape} | Test: {X_test_c.shape}')
print(f'Splits disponibles: {list(splits.keys())}')


CNC guardado OK.
Train: (14, 183) | Test: (4, 183)
Splits disponibles: ['ai4i', 'cnc', 'cnc_loo']
